## Perbandingan Hasil Forecast Antara Data 2 Tahun & 3 Tahun

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

# Load sheet Metrik_Per_Unit dari kedua file
df_2yr = pd.read_excel('Hasil Forecast 2023-2025 (ARIMA & GB & Ensemble).xlsx', sheet_name='Metrik_Per_Unit')
df_3yr = pd.read_excel('Hasil Forecast 2022-2025 (ARIMA & GB & Ensemble).xlsx', sheet_name='Metrik_Per_Unit')

# Load sheet Unit_Dikecualikan untuk referensi
excl_2yr = pd.read_excel('Hasil Forecast 2023-2025 (ARIMA & GB & Ensemble).xlsx', sheet_name='Unit_Dikecualikan')
excl_3yr = pd.read_excel('Hasil Forecast 2022-2025 (ARIMA & GB & Ensemble).xlsx', sheet_name='Unit_Dikecualikan')

print(f"Unit berhasil dimodelkan — 2 tahun train (2023-2024): {len(df_2yr)} unit")
print(f"Unit berhasil dimodelkan — 3 tahun train (2022-2024): {len(df_3yr)} unit")
print(f"Unit dikecualikan        — 2 tahun train: {len(excl_2yr)} unit")
print(f"Unit dikecualikan        — 3 tahun train: {len(excl_3yr)} unit")

Unit berhasil dimodelkan — 2 tahun train (2023-2024): 179 unit
Unit berhasil dimodelkan — 3 tahun train (2022-2024): 173 unit
Unit dikecualikan        — 2 tahun train: 60 unit
Unit dikecualikan        — 3 tahun train: 66 unit


In [2]:
# Merge berdasarkan EQUIP NAME — hanya unit yang muncul di KEDUA hasil
df_merged = df_2yr[['EQUIP NAME', 'NAMA_MASTER_TERPETAKAN',
                     'MAPE_Terpilih (%)', 'Model_Terpilih_Unit']].merge(
    df_3yr[['EQUIP NAME', 'MAPE_Terpilih (%)', 'Model_Terpilih_Unit']],
    on='EQUIP NAME',
    suffixes=('_2yr', '_3yr')
)

# Hitung selisih: negatif = 3yr lebih baik, positif = 3yr lebih buruk
df_merged['Delta_MAPE'] = df_merged['MAPE_Terpilih (%)_3yr'] - df_merged['MAPE_Terpilih (%)_2yr']

# Kategorikan
df_merged['Kategori'] = df_merged['Delta_MAPE'].apply(
    lambda x: 'Lebih Baik (3yr)' if x < 0 else ('Sama' if x == 0 else 'Lebih Buruk (3yr)')
)

print(f"Total unit yang bisa dibandingkan (muncul di kedua hasil): {len(df_merged)} unit")
print(f"\nDistribusi:")
print(df_merged['Kategori'].value_counts().to_string())
print(f"\nUnit HANYA di 2yr (tidak ada di 3yr): "
      f"{len(set(df_2yr['EQUIP NAME']) - set(df_3yr['EQUIP NAME']))} unit")
print(f"Unit HANYA di 3yr (tidak ada di 2yr): "
      f"{len(set(df_3yr['EQUIP NAME']) - set(df_2yr['EQUIP NAME']))} unit")

Total unit yang bisa dibandingkan (muncul di kedua hasil): 173 unit

Distribusi:
Kategori
Lebih Buruk (3yr)    92
Lebih Baik (3yr)     76
Sama                  5

Unit HANYA di 2yr (tidak ada di 3yr): 6 unit
Unit HANYA di 3yr (tidak ada di 2yr): 0 unit


In [3]:
df_lebih_baik = df_merged[df_merged['Kategori'] == 'Lebih Baik (3yr)'].copy()
df_lebih_baik = df_lebih_baik.sort_values('Delta_MAPE')  # paling besar perbaikannya di atas

df_lebih_baik_display = df_lebih_baik[[
    'EQUIP NAME', 'NAMA_MASTER_TERPETAKAN',
    'MAPE_Terpilih (%)_2yr', 'Model_Terpilih_Unit_2yr',
    'MAPE_Terpilih (%)_3yr', 'Model_Terpilih_Unit_3yr',
    'Delta_MAPE'
]].rename(columns={
    'MAPE_Terpilih (%)_2yr':     'MAPE 2yr (%)',
    'Model_Terpilih_Unit_2yr':   'Model 2yr',
    'MAPE_Terpilih (%)_3yr':     'MAPE 3yr (%)',
    'Model_Terpilih_Unit_3yr':   'Model 3yr',
    'Delta_MAPE':                'Selisih MAPE (%)'
}).reset_index(drop=True)

print(f"=== UNIT LEBIH BAIK DENGAN DATA 3 TAHUN ({len(df_lebih_baik)} unit) ===")
print("(Selisih MAPE negatif = MAPE 3yr lebih kecil = lebih akurat)\n")
display(df_lebih_baik_display.round(2))

=== UNIT LEBIH BAIK DENGAN DATA 3 TAHUN (76 unit) ===
(Selisih MAPE negatif = MAPE 3yr lebih kecil = lebih akurat)



,EQUIP NAME,NAMA_MASTER_TERPETAKAN,MAPE 2yr (%),Model 2yr,MAPE 3yr (%),Model 3yr,Selisih MAPE (%)
0,KALMAR 19,KALMAR 19,80.26,ARIMA,12.20,ARIMA,-68.06
1,MERPATI,MERPATI,167.80,ARIMA,105.02,ARIMA,-62.78
2,CENDET,CENDET,350.60,Gradient Boosting,309.51,Gradient Boosting,-41.09
3,L 8625 US,L 8625 US,145.94,Ensemble,106.78,ARIMA,-39.16
4,L 8060 UR (EX.L 9200 UN),L 9200 UN,164.52,ARIMA,128.86,ARIMA,-35.66
...,...,...,...,...,...,...,...
71,KALMAR1/GAJAH,KALMAR1/GAJAH,15.01,ARIMA,14.94,Ensemble,-0.07
72,KALMAR 14,KALMAR 14,12.03,Gradient Boosting,11.98,Ensemble,-0.05
73,L 8310 UQ,L 8310 UQ,10.91,Gradient Boosting,10.89,Ensemble,-0.02
74,L 8898 UQ,L 8898 UQ,35.35,ARIMA,35.34,Ensemble,-0.01


In [4]:
df_lebih_buruk = df_merged[df_merged['Kategori'] == 'Lebih Buruk (3yr)'].copy()
df_lebih_buruk = df_lebih_buruk.sort_values('Delta_MAPE', ascending=False)  # paling buruk di atas

df_lebih_buruk_display = df_lebih_buruk[[
    'EQUIP NAME', 'NAMA_MASTER_TERPETAKAN',
    'MAPE_Terpilih (%)_2yr', 'Model_Terpilih_Unit_2yr',
    'MAPE_Terpilih (%)_3yr', 'Model_Terpilih_Unit_3yr',
    'Delta_MAPE'
]].rename(columns={
    'MAPE_Terpilih (%)_2yr':     'MAPE 2yr (%)',
    'Model_Terpilih_Unit_2yr':   'Model 2yr',
    'MAPE_Terpilih (%)_3yr':     'MAPE 3yr (%)',
    'Model_Terpilih_Unit_3yr':   'Model 3yr',
    'Delta_MAPE':                'Selisih MAPE (%)'
}).reset_index(drop=True)

print(f"=== UNIT LEBIH BURUK DENGAN DATA 3 TAHUN ({len(df_lebih_buruk)} unit) ===")
print("(Selisih MAPE positif = MAPE 3yr lebih besar = kurang akurat)\n")
display(df_lebih_buruk_display.round(2))

=== UNIT LEBIH BURUK DENGAN DATA 3 TAHUN (92 unit) ===
(Selisih MAPE positif = MAPE 3yr lebih besar = kurang akurat)



,EQUIP NAME,NAMA_MASTER_TERPETAKAN,MAPE 2yr (%),Model 2yr,MAPE 3yr (%),Model 3yr,Selisih MAPE (%)
0,L 9103 NN EX. L 8781 LY,L 9103 NN EX. L 8781 LY,485.25,ARIMA,613.34,ARIMA,128.09
1,L 8251 UO,L 8251 UO,184.39,ARIMA,268.54,ARIMA,84.15
2,PALEM,PALEM,325.74,Gradient Boosting,390.48,ARIMA,64.74
3,MURAI,MURAI,59.43,ARIMA,118.50,Gradient Boosting,59.07
4,CRANE LB 80T/LUCKY DOLPHIN,CRANE LB 80T/LUCKY DOLPHIN,243.64,Gradient Boosting,275.62,Gradient Boosting,31.98
...,...,...,...,...,...,...,...
87,L 9622 US,L 9622 US,9.81,ARIMA,9.90,Ensemble,0.09
88,WERKUDORO,WERKUDORO,42.11,Gradient Boosting,42.17,Gradient Boosting,0.06
89,L 8246 US,L 8246 US,13.15,Gradient Boosting,13.21,Ensemble,0.06
90,MERAK,MERAK,26.31,ARIMA,26.36,ARIMA,0.05


In [5]:
unit_only_2yr = set(df_2yr['EQUIP NAME']) - set(df_3yr['EQUIP NAME'])
unit_only_3yr = set(df_3yr['EQUIP NAME']) - set(df_2yr['EQUIP NAME'])

print("=== UNIT YANG HANYA BERHASIL DIMODELKAN DI 2 TAHUN (tidak lolos filter di 3 tahun) ===")
df_only_2yr = df_2yr[df_2yr['EQUIP NAME'].isin(unit_only_2yr)][
    ['EQUIP NAME', 'NAMA_MASTER_TERPETAKAN', 'MAPE_Terpilih (%)', 'Model_Terpilih_Unit']
].sort_values('EQUIP NAME').reset_index(drop=True)
display(df_only_2yr)

print(f"\n=== UNIT YANG HANYA BERHASIL DIMODELKAN DI 3 TAHUN (tidak lolos filter di 2 tahun) ===")
df_only_3yr = df_3yr[df_3yr['EQUIP NAME'].isin(unit_only_3yr)][
    ['EQUIP NAME', 'NAMA_MASTER_TERPETAKAN', 'MAPE_Terpilih (%)', 'Model_Terpilih_Unit']
].sort_values('EQUIP NAME').reset_index(drop=True)
display(df_only_3yr)

# Cek alasan eksklusi
if not unit_only_2yr.empty if isinstance(unit_only_2yr, pd.Series) else len(unit_only_2yr) > 0:
    print("\nAlasan unit 2yr tidak muncul di hasil 3yr:")
    display(excl_3yr[excl_3yr['EQUIP NAME'].isin(unit_only_2yr)][
        ['EQUIP NAME', 'Alasan']
    ].reset_index(drop=True))

=== UNIT YANG HANYA BERHASIL DIMODELKAN DI 2 TAHUN (tidak lolos filter di 3 tahun) ===


,EQUIP NAME,NAMA_MASTER_TERPETAKAN,MAPE_Terpilih (%),Model_Terpilih_Unit
0,CAMAR,CAMAR,18.00,Ensemble
1,JATAYU,JATAYU,11.86,ARIMA
2,KUDA PUTIH,KUDA PUTIH,25.65,ARIMA
3,L 9049 UQ,L 9049 UQ,19.01,Gradient Boosting
4,L 9061 UR,L 9061 UR,17.31,ARIMA
5,TOP LOADER MITS/CENDRAWASIH,TOP LOADER MITS/CENDRAWASIH,108.16,Gradient Boosting



=== UNIT YANG HANYA BERHASIL DIMODELKAN DI 3 TAHUN (tidak lolos filter di 2 tahun) ===


,EQUIP NAME,NAMA_MASTER_TERPETAKAN,MAPE_Terpilih (%),Model_Terpilih_Unit



Alasan unit 2yr tidak muncul di hasil 3yr:


,EQUIP NAME,Alasan
0,CAMAR,HM/LITER = 0 selama 1 tahun penuh di Data Lati...
1,JATAYU,HM/LITER = 0 selama 1 tahun penuh di Data Lati...
2,KUDA PUTIH,HM/LITER = 0 selama 1 tahun penuh di Data Lati...
3,L 9049 UQ,HM/LITER = 0 selama 1 tahun penuh di Data Lati...
4,L 9061 UR,HM/LITER = 0 selama 1 tahun penuh di Data Lati...
5,TOP LOADER MITS/CENDRAWASIH,HM/LITER = 0 selama 1 tahun penuh di Data Lati...


In [ ]:
fig = px.scatter(
    df_merged,
    x='MAPE_Terpilih (%)_2yr',
    y='MAPE_Terpilih (%)_3yr',
    color='Kategori',
    color_discrete_map={
        'Lebih Baik (3yr)':  '#2ecc71',
        'Lebih Buruk (3yr)': '#e74c3c',
        'Sama':              '#95a5a6'
    },
    hover_data=['EQUIP NAME', 'NAMA_MASTER_TERPETAKAN',
                'Model_Terpilih_Unit_2yr', 'Model_Terpilih_Unit_3yr',
                'Delta_MAPE'],
    title='Perbandingan MAPE: Train 2 Tahun (2023-2024) vs Train 3 Tahun (2022-2024)',
    labels={
        'MAPE_Terpilih (%)_2yr': 'MAPE Train 2 Tahun (%)',
        'MAPE_Terpilih (%)_3yr': 'MAPE Train 3 Tahun (%)'
    }
)

# Garis diagonal y=x sebagai referensi: di bawah garis = 3yr lebih baik
max_val = max(df_merged['MAPE_Terpilih (%)_2yr'].max(),
              df_merged['MAPE_Terpilih (%)_3yr'].max()) * 1.05
fig.add_shape(
    type='line', x0=0, y0=0, x1=max_val, y1=max_val,
    line=dict(color='gray', dash='dash', width=1.5)
)
fig.add_annotation(
    x=max_val * 0.75, y=max_val * 0.82,
    text="y = x  (tidak ada perubahan)",
    showarrow=False, font=dict(color='gray', size=11)
)

fig.update_traces(marker=dict(size=8, opacity=0.75))
fig.update_layout(height=600, legend_title_text='Hasil Perbandingan')
fig.show()

: 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Kiri: 15 unit yang paling banyak membaik
top_baik = df_lebih_baik.nsmallest(15, 'Delta_MAPE')
axes[0].barh(top_baik['EQUIP NAME'], top_baik['Delta_MAPE'].abs(), color='#2ecc71')
axes[0].set_title('Top 15 Unit — Paling Banyak Membaik\n(Data 3 Tahun vs 2 Tahun)', fontweight='bold')
axes[0].set_xlabel('Penurunan MAPE (%)')
axes[0].invert_yaxis()
for i, (val, mape2, mape3) in enumerate(zip(
    top_baik['Delta_MAPE'].abs(),
    top_baik['MAPE_Terpilih (%)_2yr'],
    top_baik['MAPE_Terpilih (%)_3yr']
)):
    axes[0].text(val + 0.3, i, f'{mape2:.1f}% → {mape3:.1f}%', va='center', fontsize=8)

# Kanan: 15 unit yang paling banyak memburuk
top_buruk = df_lebih_buruk.nlargest(15, 'Delta_MAPE')
axes[1].barh(top_buruk['EQUIP NAME'], top_buruk['Delta_MAPE'], color='#e74c3c')
axes[1].set_title('Top 15 Unit — Paling Banyak Memburuk\n(Data 3 Tahun vs 2 Tahun)', fontweight='bold')
axes[1].set_xlabel('Kenaikan MAPE (%)')
axes[1].invert_yaxis()
for i, (val, mape2, mape3) in enumerate(zip(
    top_buruk['Delta_MAPE'],
    top_buruk['MAPE_Terpilih (%)_2yr'],
    top_buruk['MAPE_Terpilih (%)_3yr']
)):
    axes[1].text(val + 0.3, i, f'{mape2:.1f}% → {mape3:.1f}%', va='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
fig = px.histogram(
    df_merged,
    x='Delta_MAPE',
    nbins=40,
    color='Kategori',
    color_discrete_map={
        'Lebih Baik (3yr)':  '#2ecc71',
        'Lebih Buruk (3yr)': '#e74c3c',
        'Sama':              '#95a5a6'
    },
    title='Distribusi Selisih MAPE (MAPE 3yr − MAPE 2yr)<br>'
          '<sup>Nilai negatif = data 3 tahun lebih akurat</sup>',
    labels={'Delta_MAPE': 'Selisih MAPE (%) = MAPE 3yr − MAPE 2yr'}
)

fig.add_vline(x=0, line_dash='dash', line_color='black',
              annotation_text='Tidak ada perubahan',
              annotation_position='top right')
fig.add_vline(x=df_merged['Delta_MAPE'].mean(),
              line_dash='dot', line_color='navy',
              annotation_text=f"Rata-rata: {df_merged['Delta_MAPE'].mean():.2f}%",
              annotation_position='top left')

fig.update_layout(height=450, bargap=0.05)
fig.show()

In [ ]:
print("=" * 65)
print(" RINGKASAN PERBANDINGAN HASIL FORECAST ".center(65))
print("=" * 65)
print(f"Total unit yang dapat dibandingkan          : {len(df_merged)} unit")
print(f"  Lebih baik dengan data 3 tahun            : {len(df_lebih_baik)} unit "
      f"({len(df_lebih_baik)/len(df_merged)*100:.1f}%)")
print(f"  Lebih buruk dengan data 3 tahun           : {len(df_lebih_buruk)} unit "
      f"({len(df_lebih_buruk)/len(df_merged)*100:.1f}%)")

print(f"\nRata-rata MAPE seluruh unit (yang bisa dibandingkan):")
print(f"  Train 2 tahun (2023-2024) : {df_merged['MAPE_Terpilih (%)_2yr'].mean():.2f}%")
print(f"  Train 3 tahun (2022-2024) : {df_merged['MAPE_Terpilih (%)_3yr'].mean():.2f}%")
print(f"  Rata-rata Delta MAPE      : {df_merged['Delta_MAPE'].mean():.2f}% "
      f"({'membaik' if df_merged['Delta_MAPE'].mean() < 0 else 'memburuk'})")

print(f"\nUnit yang tidak bisa dibandingkan:")
print(f"  Hanya ada di hasil 2 tahun                : {len(unit_only_2yr)} unit")
print(f"  Hanya ada di hasil 3 tahun                : {len(unit_only_3yr)} unit")

print(f"\nPerbaikan terbesar dengan data 3 tahun:")
best_row = df_merged.loc[df_merged['Delta_MAPE'].idxmin()]
print(f"  {best_row['EQUIP NAME']:30} : {best_row['MAPE_Terpilih (%)_2yr']:.2f}% → "
      f"{best_row['MAPE_Terpilih (%)_3yr']:.2f}% (Δ {best_row['Delta_MAPE']:.2f}%)")

print(f"\nPenurunan terbesar dengan data 3 tahun:")
worst_row = df_merged.loc[df_merged['Delta_MAPE'].idxmax()]
print(f"  {worst_row['EQUIP NAME']:30} : {worst_row['MAPE_Terpilih (%)_2yr']:.2f}% → "
      f"{worst_row['MAPE_Terpilih (%)_3yr']:.2f}% (Δ +{worst_row['Delta_MAPE']:.2f}%)")
print("=" * 65)